여기서 torchmetrics(mAP 계산용)이랑 pycocotools(COCO 형식 처리용)를 설치함. 코랩 기본 환경엔 없어서 따로 깔아줘야 함.

In [ ]:
!pip install -q torchmetrics pycocotools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.3 MB/s eta 0:00:00


필요한 라이브러리 불러오는 부분. torch/torchvision은 모델·학습용이고, tv_tensors랑 v2는 최신 torchvision 방식으로 이미지랑 박스를 같이 변형(resize 등)할 때 씀. kagglehub는 캐글 대회 데이터 받을 때 씀.

In [ ]:
#@title Import Module
import os
import cv2
import csv
import glob
import json
import numpy as np
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
import xml.etree.ElementTree as ET

import torch
import torchvision

from torchvision import tv_tensors
from torchvision.transforms import v2
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from PIL import Image
from tqdm import tqdm
import kagglehub

GPU 있으면 cuda, 없으면 cpu로 잡히게 하는 셀. 코랩에서 런타임을 GPU로 안 바꾸고 실행하면 여기서 cpu로 찍히니까 꼭 확인해야 함(런타임 > 런타임 유형 변경 > GPU).

In [ ]:
#@title Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


캐글 대회 데이터를 받으려면 인증이 필요해서 넣은 셀. Colab Secrets에 KAGGLE_API_TOKEN을 저장해두고 여기서 불러오는 방식. 왼쪽 열쇠 아이콘에서 이 노트북에 접근 허용 체크를 안 해두면 SecretNotFoundError가 남.

In [ ]:
#@title Kaggle 인증
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")


kagglehub로 대회 데이터를 다운받고, train/test 이미지 경로랑 annotation(json) 경로를 각각 리스트로 만들어두는 셀. 이 뒤에 나오는 셀들은 다 이 image_dir / annotation_dir을 기준으로 돌아감.

In [ ]:
#@title Image, Annotation 경로 설정
#path = r"/Users/codeit/Desktop/Vault_home/💼 Projects(Work)/Sprint AI_14기/workspace/프로젝트_초급/sprint_ai_project1_data"
path = kagglehub.competition_download("ai14-level-project")
path = os.path.join(path, "sprint_ai_project1_data")
print(path)

image_dir = glob.glob(os.path.join(path, "train_images", "*.png"))

annotation_dir = glob.glob(
    os.path.join(path, "train_annotations", "**", "*.json"),
    recursive=True
)

print(f"Train 이미지 수 : {len(image_dir)}, Annotation 수 : {len(annotation_dir)}")

100%|██████████| 1.79G/1.79G [00:26<00:00, 72.3MB/s]

Extracting files...


/root/.cache/kagglehub/competitions/ai14-level-project/sprint_ai_project1_data
Train 이미지 수 : 232, Annotation 수 : 763


In [ ]:
# #@title Image-Annotation Mapping

# # Train 이미지에 해당하지 않는 Annotation 개수 확인
# annotation_not_in_train_image = []
# for image_path in image_dir:
#     image_name = os.path.basename(image_path)
#     annotation_name = image_name.replace(".png", ".json")
#     if annotation_name not in [os.path.basename(annotation_path) for annotation_path in annotation_dir]:
#         annotation_not_in_train_image.append(image_name)

# print(f"Annotation에 해당하지 않는 Train 이미지 개수: {len(annotation_not_in_train_image)}")

# # Annotation 에 없는 Train 이미지 개수 확인
# train_image_not_in_annotation = []
# for annotation_path in annotation_dir:
#     annotation_name = os.path.basename(annotation_path)
#     image_name = annotation_name.replace(".json", ".png")
#     if image_name not in [os.path.basename(image_path) for image_path in image_dir]:
#         train_image_not_in_annotation.append(annotation_name)

# print(f"Train 이미지에 해당하지 않는 Annotation 개수: {len(train_image_not_in_annotation)}")

# # Train 이미지와 Annotation 매핑 가능한 데이터 수
# image_annotation_mapping = {}
# for image_path in image_dir:
#     image_name = os.path.basename(image_path)
#     annotation_name = image_name.replace(".png", ".json")
#     if annotation_name in [os.path.basename(annotation_path) for annotation_path in annotation_dir]:
#         image_annotation_mapping[image_name] = annotation_name

# print(f"Train 이미지와 Annotation 매핑 가능한 데이터 수: {len(image_annotation_mapping)}")

# # Image - Annotation Mapping을 위한 코드
# image_names = [os.path.basename(image_path) for image_path in image_dir]
# annotation_names = [os.path.basename(annotation_path) for annotation_path in annotation_dir]

# image_annotation_mapping_process = {}
# for image_name in image_names:
#     annotation_name = image_name.replace(".png", ".json")
#     if annotation_name in annotation_names:
#         image_annotation_mapping_process[image_name] = annotation_name


이상하다? Train 이미지는 232개인데, Annotation은 763개인데 Train 이미지에 해당하지 않는 Annotation의 개수가 0개라는게 말이 안된다고 생각했습니다.

그래서 중복되는 annotation이 있는지 확인하는게 다음 코드입니다.

In [ ]:
# #@title Annotation 경로 매핑 (중복 대응)
# annotation_path_by_stem = {}
# duplicate_mismatch = []

# for ann_path in annotation_dir:
#     stem = os.path.splitext(os.path.basename(ann_path))[0]
#     if stem in annotation_path_by_stem:
#         # 중복 파일이 내용까지 같은지 확인
#         with open(annotation_path_by_stem[stem], "r", encoding="utf-8") as f1, open(ann_path, "r", encoding="utf-8") as f2:
#             if f1.read() != f2.read():
#                 duplicate_mismatch.append(stem)
#     else:
#         annotation_path_by_stem[stem] = ann_path

# print(f"고유 annotation stem 수: {len(annotation_path_by_stem)}")
# print(f"내용이 다른 중복 파일 수: {len(duplicate_mismatch)}")
# if duplicate_mismatch:
#     print("확인 필요:", duplicate_mismatch[:10])

In [ ]:
# #@title 중복 annotation 내용 진단
# from collections import defaultdict

# stem_to_paths = defaultdict(list)
# for ann_path in annotation_dir:
#     stem = os.path.splitext(os.path.basename(ann_path))[0]
#     stem_to_paths[stem].append(ann_path)

# # 확인 필요 목록 중 하나 골라서 상세 비교
# sample_stem = "K-003351-016232-033880_0_2_0_2_75_000_200"
# sample_paths = stem_to_paths[sample_stem]

# print(f"'{sample_stem}' 관련 파일 개수: {len(sample_paths)}")
# for p in sample_paths:
#     with open(p, "r", encoding="utf-8") as f:
#         data = json.load(f)
#     ann_summaries = [(a["id"], a["category_id"], a["bbox"]) for a in data["annotations"]]
#     cat_names = [c["name"] for c in data["categories"]]
#     print("---")
#     print("경로:", p)
#     print("categories:", cat_names)
#     print("annotations (id, category_id, bbox):", ann_summaries)

In [ ]:
# #@title Annotation 경로 매핑 (stem -> 여러 파일)
# from collections import defaultdict

# stem_to_paths = defaultdict(list)
# for ann_path in annotation_dir:
#     stem = os.path.splitext(os.path.basename(ann_path))[0]
#     stem_to_paths[stem].append(ann_path)

# avg_per_stem = sum(len(v) for v in stem_to_paths.values()) / len(stem_to_paths)
# print(f"고유 이미지 수: {len(stem_to_paths)}, 이미지당 평균 annotation 파일 수: {avg_per_stem:.2f}")

이미지 파일명(stem)이랑 annotation 파일들을 서로 연결하는 부분. 이미지 1개당 annotation json이 여러 개(알약 인스턴스별로 하나씩) 있어서, stem_to_paths는 하나의 stem에 여러 json 경로를 리스트로 묶어둔 딕셔너리임. valid_stems는 이미지랑 annotation이 둘 다 있는 것만 남긴 목록.

In [ ]:
#Cell 4 수정(1:1 mapping 코드 -> 1:n mapping 코드)
#@title Image-Annotation Mapping (stem 기준)

from collections import defaultdict

# 이미지 stem -> 이미지 경로
image_stem_to_path = {
    os.path.splitext(os.path.basename(image_path))[0]: image_path
    for image_path in image_dir
}

# 이미지 stem -> 여러 annotation JSON 경로
stem_to_paths = defaultdict(list)

for annotation_path in annotation_dir:
    stem = os.path.splitext(os.path.basename(annotation_path))[0]
    stem_to_paths[stem].append(annotation_path)

# 이미지와 annotation이 모두 존재하는 stem만 사용
valid_stems = [
    stem
    for stem in stem_to_paths
    if stem in image_stem_to_path
]

# 매핑되지 않는 데이터 확인
images_without_annotation = (
    set(image_stem_to_path) - set(stem_to_paths)
)

annotations_without_image = (
    set(stem_to_paths) - set(image_stem_to_path)
)

print(f"Annotation이 없는 Train 이미지 수: {len(images_without_annotation)}")
print(f"Train 이미지가 없는 Annotation stem 수: {len(annotations_without_image)}")
print(f"매핑 가능한 이미지 수: {len(valid_stems)}")

Annotation이 없는 Train 이미지 수: 0
Train 이미지가 없는 Annotation stem 수: 0
매핑 가능한 이미지 수: 232


위에서 만든 stem_to_paths 기준으로 이미지 하나당 annotation이 평균 몇 개씩 붙어있는지 확인하는 셀. 여기서 나온 평균값이 763(annotation 총개수) / 232(이미지 개수)랑 맞아떨어지는지 체크하는 용도.

In [ ]:
#@title Annotation 경로 매핑 (stem -> 여러 파일)
avg_per_stem = sum(len(v) for v in stem_to_paths.values()) / len(stem_to_paths)

print(
    f"고유 이미지 수: {len(stem_to_paths)}, "
    f"이미지당 평균 annotation 파일 수: {avg_per_stem:.2f}"
)

고유 이미지 수: 232, 이미지당 평균 annotation 파일 수: 3.29


#수정 사항

기존 코드 흐름은 이미지 하나당 어노테이션 하나로 매핑하였으나 이후 중복된 어노테이션 발견 후 이미지 하나당 여러개의 어노테이션을 매핑하는 코드로 바꿨습니다. 이걸 효울성을 위해 처음부터 이미지 하나당 어노테이션 하나로 매핑하는 형태로 수정했습니다.

In [ ]:
# #@title 폴더별 partial annotation 병합
# def load_merged_annotation(paths):
#     img_info = None
#     boxes, labels, areas, iscrowd = [], [], [], []

#     for p in paths:
#         with open(p, "r", encoding="utf-8") as f:
#             data = json.load(f)

#         if img_info is None:
#             img_info = data["images"][0]  # width/height/file_name은 모든 폴더에서 동일하다고 가정

#         for ann in data["annotations"]:
#             boxes.append(ann["bbox"])       # [x, y, w, h]
#             labels.append(ann["category_id"])
#             areas.append(ann["area"])
#             iscrowd.append(ann.get("iscrowd", 0))

#     return img_info, boxes, labels, areas, iscrowd

제일 오래 걸리는 셀. 각 이미지(stem)마다 딸려있는 annotation json들을 전부 열어서 bbox·category_id를 하나의 image_records 딕셔너리로 합쳐줌. 파일을 763개나 하나씩 열어야 해서 코랩에서 몇 분 걸릴 수 있음(실제로 여기서 5~7분 정도 걸렸음, 에러 아니니까 그냥 기다리면 됨).

In [ ]:
#@title 이미지 단위 Annotation 병합

def build_image_records(valid_stems, stem_to_paths):
    image_records = {}
    category_id_to_name = {}

    for stem in valid_stems:
        img_info = None
        annotations = []

        for annotation_path in stem_to_paths[stem]:
            with open(annotation_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # 이미지 정보는 같은 stem의 JSON끼리 동일하므로 처음 한 번만 저장
            if img_info is None:
                img_info = data["images"][0]

            # 클래스 정보 저장
            for category in data["categories"]:
                category_id_to_name[category["id"]] = category["name"]

            # 같은 이미지에 속한 annotation들을 하나로 병합
            for ann in data["annotations"]:
                annotations.append({
                    "bbox": ann["bbox"],
                    "category_id": ann["category_id"],
                    "area": ann["area"],
                    "iscrowd": ann.get("iscrowd", 0)
                })

        image_records[stem] = {
            "file_name": img_info["file_name"],
            "width": img_info["width"],
            "height": img_info["height"],
            "annotations": annotations
        }

    return image_records, category_id_to_name


image_records, category_id_to_name = build_image_records(
    valid_stems,
    stem_to_paths
)

print(f"이미지 record 수: {len(image_records)}")
print(f"클래스 수: {len(category_id_to_name)}")

이미지 record 수: 232
클래스 수: 56


여기서 진짜 잘못된 라벨을 걸러냄. bbox의 x+w나 y+h가 이미지 크기(W, H)를 넘어가거나 좌표가 음수면 이상한 라벨로 보고 제외하는 방식. 실제로 이 조건에 걸린 게 763개 중 딱 1개 있었음.

In [ ]:
#@title 잘못된 라벨(bbox 범위 이탈) 제외
def filter_invalid_annotations(image_records):
    cleaned = {}
    removed = []

    for stem, rec in image_records.items():
        W, H = rec["width"], rec["height"]
        valid_anns = []

        for ann in rec["annotations"]:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0 or x < 0 or y < 0 or x + w > W or y + h > H:
                removed.append((stem, ann["bbox"]))
                continue
            valid_anns.append(ann)

        if valid_anns:
            new_rec = dict(rec)
            new_rec["annotations"] = valid_anns
            cleaned[stem] = new_rec
        else:
            removed.append((stem, "이미지 전체 제외(유효 annotation 없음)"))

    print(f"제외된 annotation/이미지: {len(removed)}건")
    for stem, info in removed:
        print(f"  {stem}: {info}")

    return cleaned


image_records = filter_invalid_annotations(image_records)


제외된 annotation/이미지: 1건
  K-003351-016262-018357_0_2_0_2_75_000_200: [6567, 625, 311, 315]


정제된 image_records에서 이미지 5장만 랜덤으로 뽑아서 bbox를 실제로 그려보는 셀. bbox는 [x, y, w, h] 형태로 저장돼있는데 그림을 그리려면 좌상단/우하단 좌표(x1, y1, x2, y2)가 필요해서 x2=x+w, y2=y+h로 바꿔서 그림 — 이 변환을 반대로 하면 박스가 엉뚱하게 그려짐(전에 한 번 실수했던 부분).

In [ ]:
#@title Train 이미지 + Bounding Box 시각화 (정제된 image_records 기준)
import random

n_samples = 5
sample_stems = random.sample(list(image_records.keys()), n_samples)

fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 5))

for ax, stem in zip(axes, sample_stems):
    rec = image_records[stem]
    img = cv2.imread(os.path.join(path, "train_images", rec["file_name"]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for ann in rec["annotations"]:
        x, y, w, h = ann["bbox"]
        x1, y1, x2, y2 = int(x), int(y), int(x + w), int(y + h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(img, str(ann["category_id"]), (x1, max(y1 - 10, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    ax.imshow(img)
    ax.set_title(stem, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


#수정 사항

기존 코드: Dataset에서 이미지 하나 꺼냄 → 그때 JSON 파일 여러 개 다시 열기 → 병합

수정 코드: 학습 시작 전에 JSON 전부 한 번 읽기 → image_records에 저장  Dataset → image_records에서 바로 가져오기

속도/효율 개선, 코드 복잡성 개선을 위해 수정했습니다.

학습 이미지들 해상도가 다 같은지 확인하는 셀. 200장만 먼저 봤는데 전부 (976, 1280)로 동일해서 전수 조사는 안 해도 됐음.

In [ ]:
#@title 이미지 해상도 파악
sizes = set()
for img_path in image_dir[:50]:  # 200장이면 전수 조사해도 됨: image_dir 전체로
    with Image.open(img_path) as im:
        sizes.add(im.size)  # (width, height)

print(f"고유 해상도 종류 수: {len(sizes)}")
print(sizes if len(sizes) < 10 else list(sizes)[:10])

고유 해상도 종류 수: 1
{(976, 1280)}


이미지를 모델에 넣기 전에 텐서로 바꾸고 크기를 통일시키는 전처리 부분. train_transformer는 학습용, val_test_transformer는 검증/테스트용인데 지금은 둘 다 랜덤 augmentation 없이 Resize만 들어가 있음.

In [ ]:
#@title Transformer
train_transformer = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.Resize((640, 640)),
])

val_test_transformer = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.Resize((640, 640)),
])

In [ ]:
# #@title 클래스(category) 매핑 생성
# import json

# def build_category_mapping(annotation_paths):
#     category_id_to_name = {}
#     for path in annotation_paths:
#         with open(path, "r", encoding="utf-8") as f:
#             data = json.load(f)
#         for cat in data["categories"]:
#             category_id_to_name[cat["id"]] = cat["name"]
#     return category_id_to_name

# category_id_to_name = build_category_mapping(annotation_dir)  # annotation_dir: glob으로 뽑은 json 경로 리스트
# sorted_ids = sorted(category_id_to_name.keys())

# category_to_label = {cid: i + 1 for i, cid in enumerate(sorted_ids)}  # 0 = background
# label_to_category = {v: k for k, v in category_to_label.items()}
# classes = ["background"] + [category_id_to_name[cid] for cid in sorted_ids]
# num_classes = len(classes)

# print(f"클래스 수(background 포함): {num_classes}")

원본 category_id(약품 코드)를 모델이 쓸 수 있는 0~num_classes 범위의 라벨 번호로 바꿔주는 매핑을 만드는 셀. category_to_label / label_to_category 이 두 딕셔너리는 학습할 때랑, 나중에 예측 결과를 다시 원래 category_id로 되돌릴 때(제출 파일 만들 때) 둘 다 필요함.

In [ ]:
#@title 클래스(category) 매핑 생성
sorted_ids = sorted(category_id_to_name.keys())

category_to_label = {
    cid: i + 1
    for i, cid in enumerate(sorted_ids)
}

label_to_category = {
    v: k
    for k, v in category_to_label.items()
}

classes = ["background"] + [
    category_id_to_name[cid]
    for cid in sorted_ids
]

num_classes = len(classes)

print(f"클래스 수(background 포함): {num_classes}")

클래스 수(background 포함): 57


#수정 사항
build_image_records 함수에서 전체 json파일을 읽는 과정을 할때 클래스 매핑작업을 추가했으니 이후에 진행하면 효율성에 떨어지므로 다시 매핑하는 작업을 삭제했습니다.

In [ ]:
# #@title Pill Dataset 정의 (병합 버전)
# class PillDataset(Dataset):
#     def __init__(self, image_dir, stem_to_paths, stems, category_to_label, transforms=None):
#         self.image_dir = image_dir
#         self.stem_to_paths = stem_to_paths
#         self.stems = stems
#         self.category_to_label = category_to_label
#         self.transforms = transforms

#     def __len__(self):
#         return len(self.stems)

#     def __getitem__(self, idx):
#         stem = self.stems[idx]
#         paths = self.stem_to_paths[stem]

#         img_info, raw_boxes, raw_labels, areas, iscrowd = load_merged_annotation(paths)

#         image = Image.open(os.path.join(self.image_dir, img_info["file_name"])).convert("RGB")
#         w, h = img_info["width"], img_info["height"]

#         boxes = [[x, y, x + bw, y + bh] for x, y, bw, bh in raw_boxes]
#         labels = [self.category_to_label[cid] for cid in raw_labels]

#         image = tv_tensors.Image(image)
#         boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(h, w))

#         target = {
#             "boxes": boxes,
#             "labels": torch.as_tensor(labels, dtype=torch.int64),
#             "image_id": torch.tensor([idx]),
#             "area": torch.as_tensor(areas, dtype=torch.float32),
#             "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64),
#         }

#         if self.transforms:
#             image, target = self.transforms(image, target)

#         return image, target

PyTorch Dataset 클래스 정의하는 셀. __getitem__에서 이미지를 열고 image_records에 있는 bbox/label을 텐서로 바꿔서 돌려줌. boxes를 tv_tensors.BoundingBoxes로 감싸는 이유는, 이후 Resize 같은 transform을 적용할 때 박스 좌표도 이미지 크기 변화에 맞춰 자동으로 같이 바뀌게 하려는 거임.

In [ ]:
#@title Pill Dataset 정의

class PillDataset(Dataset):
    def __init__(
        self,
        image_dir,
        image_records,
        stems,
        category_to_label,
        transforms=None
    ):
        self.image_dir = image_dir
        self.image_records = image_records
        self.stems = stems
        self.category_to_label = category_to_label
        self.transforms = transforms

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]

        # 미리 병합해둔 annotation 정보 가져오기
        record = self.image_records[stem]

        image = Image.open(
            os.path.join(self.image_dir, record["file_name"])
        ).convert("RGB")

        w = record["width"]
        h = record["height"]

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in record["annotations"]:
            x, y, bw, bh = ann["bbox"]

            boxes.append([x, y, x + bw, y + bh])
            labels.append(
                self.category_to_label[ann["category_id"]]
            )
            areas.append(ann["area"])
            iscrowd.append(ann["iscrowd"])

        image = tv_tensors.Image(image)

        boxes = tv_tensors.BoundingBoxes(
            boxes,
            format="XYXY",
            canvas_size=(h, w)
        )

        target = {
            "boxes": boxes,
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
            "area": torch.as_tensor(areas, dtype=torch.float32),
            "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64),
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target

In [ ]:
# #@title Dataset ~ DataLoader
# stems = list(stem_to_paths.keys())
# train_stems, valid_stems = train_test_split(stems, test_size=0.2, random_state=42)

# train_dataset = PillDataset(image_dir=os.path.join(path, "train_images"), stem_to_paths=stem_to_paths,
#                              stems=train_stems, category_to_label=category_to_label, transforms=train_transformer)
# valid_dataset = PillDataset(image_dir=os.path.join(path, "train_images"), stem_to_paths=stem_to_paths,
#                              stems=valid_stems, category_to_label=category_to_label, transforms=val_test_transformer)

# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
# val_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

In [ ]:
# #@title Dataset ~ DataLoader
# stems = list(stem_to_paths.keys())
# train_stems, valid_stems = train_test_split(stems, test_size=0.2, random_state=42)

# train_dataset = PillDataset(
#     image_dir=os.path.join(path, "train_images"),
#     image_records=image_records,
#     stems=train_stems,
#     category_to_label=category_to_label,
#     transforms=train_transformer
# )
# valid_dataset = PillDataset(
#     image_dir=os.path.join(path, "train_images"),
#     image_records=image_records,
#     stems=valid_stems,
#     category_to_label=category_to_label,
#     transforms=val_test_transformer
# )

# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
# val_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

#수정 사항

PillDataset의 입력을 기존 stem_to_paths에서 image_records로 바꿨기 때문에 stem_to_paths=stem_to_paths를 image_records=image_records로 바꿨습니다.

In [ ]:
#@title 클래스별 인스턴스 수 확인 (데이터 불균형 파악)
from collections import Counter

class_instance_count = Counter()
for record in image_records.values():
    for ann in record["annotations"]:
        class_instance_count[ann["category_id"]] += 1

total_instances = sum(class_instance_count.values())
sorted_counts = sorted(class_instance_count.items(), key=lambda x: x[1])

RARE_THRESHOLD = 5
n_rare = sum(1 for _, c in sorted_counts if c <= RARE_THRESHOLD)
top_id, top_count = sorted_counts[-1]
bottom_id, bottom_count = sorted_counts[0]

print(f"전체 클래스 수: {len(class_instance_count)}, 전체 인스턴스 수: {total_instances}")
print(f"인스턴스 {RARE_THRESHOLD}개 이하인 희귀 클래스: {n_rare}개")
print(f"최다 클래스: {category_id_to_name[top_id]} ({top_count}회, {top_count/total_instances:.1%})")
print(f"최소 클래스: {category_id_to_name[bottom_id]} ({bottom_count}회)")


전체 클래스 수: 56, 전체 인스턴스 수: 762
인스턴스 5개 이하인 희귀 클래스: 18개
최다 클래스: 일양하이트린정 2mg (153회, 20.1%)
최소 클래스: 제미메트서방정 50/1000mg (3회)


여기가 클래스 불균형 대응하는 핵심 부분. 인스턴스 5개 이하인 희귀 클래스는 최대한 train 쪽에 몰아넣고 val에는 최소 1장만 남기고, WeightedRandomSampler로 학습 중에도 희귀 클래스 이미지가 더 자주 뽑히게 함. 그냥 8:2로 랜덤 split하면 희귀 클래스가 val에만 몰리거나 train에 아예 하나도 없을 수 있어서 이렇게 나눔.

In [ ]:
#@title Dataset ~ DataLoader (클래스 불균형 대응: split 보정 + WeightedRandomSampler)
stems = list(stem_to_paths.keys())

# 이미지(stem)별로 등장하는 클래스 집합
stem_to_classes = {
    stem: {ann["category_id"] for ann in image_records[stem]["annotations"]}
    for stem in stems
}

rare_classes = {cid for cid, c in class_instance_count.items() if c <= RARE_THRESHOLD}
rare_stems = [s for s in stems if stem_to_classes[s] & rare_classes]
rare_stem_set = set(rare_stems)
common_stems = [s for s in stems if s not in rare_stem_set]

# common 이미지는 기존처럼 8:2로 split
common_train, common_valid = train_test_split(common_stems, test_size=0.2, random_state=42)

# 희귀 클래스 이미지는 val에 최소 1장만 남기고 대부분 train으로 (val 크기가 부족하면 전부 train)
if len(rare_stems) > 1:
    rare_valid_size = max(1, round(len(rare_stems) * 0.2))
    rare_train, rare_valid = train_test_split(
        rare_stems, test_size=rare_valid_size, random_state=42
    )
else:
    rare_train, rare_valid = rare_stems, []

train_stems = common_train + rare_train
valid_stems = common_valid + rare_valid

print(f"train {len(train_stems)}장 / valid {len(valid_stems)}장 "
      f"(희귀 클래스 포함 이미지 {len(rare_stems)}장 중 valid에 {len(rare_valid)}장 배정)")

train_dataset = PillDataset(
    image_dir=os.path.join(path, "train_images"),
    image_records=image_records,
    stems=train_stems,
    category_to_label=category_to_label,
    transforms=train_transformer
)
valid_dataset = PillDataset(
    image_dir=os.path.join(path, "train_images"),
    image_records=image_records,
    stems=valid_stems,
    category_to_label=category_to_label,
    transforms=val_test_transformer
)

# 이미지별 샘플링 가중치: 이미지에 포함된 클래스 중 가장 희귀한 클래스의 역빈도
image_weights = [
    1.0 / min(class_instance_count[c] for c in stem_to_classes[stem])
    for stem in train_stems
]

sampler = torch.utils.data.WeightedRandomSampler(
    weights=image_weights,
    num_samples=len(train_stems),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=4, sampler=sampler, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


train 185장 / valid 47장 (희귀 클래스 포함 이미지 39장 중 valid에 8장 배정)


In [ ]:
# model = torchvision.models.detection.ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT).to(device)
# in_channels = det_utils.retrieve_out_channels(model.backbone, (300, 300))
# num_anchors = model.anchor_generator.num_anchors_per_location()
# model.head.classification_head = SSDClassificationHead(in_channels, num_anchors, num_classes).to(device)

RetinaNet(ResNet50 FPN v2) 모델을 만드는 셀. weights=None으로 둔 이유는, torchvision이 주는 pretrained weight가 COCO 클래스 개수(91개)에 맞춰져 있어서 우리 클래스 개수(57개)로 바꾸려면 그대로 못 쓰기 때문. 대신 weights_backbone=ResNet50_Weights.DEFAULT로 backbone(이미지 특징 추출 부분)만 ImageNet pretrained로 가져오고, head는 새로 학습시킴.

In [ ]:
#@title 모델 정의 (RetinaNet ResNet50 FPN v2)
model = retinanet_resnet50_fpn_v2(
    weights=None,
    weights_backbone=ResNet50_Weights.DEFAULT,
    num_classes=num_classes
).to(device)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 182MB/s]


In [ ]:
# #@title Define Optimizer & Scheduler
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=0.0005)
# lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

옵티마이저를 SGD에서 Adam으로 바꾼 셀(원래 인호님 SSD 코드는 SGD였음). lr_scheduler는 3 epoch마다 학습률을 1/10로 줄여서, 학습 후반부에 너무 크게 움직이지 않고 세밀하게 수렴하게 함.

In [ ]:
#@title Define Optimizer & Scheduler (SGD → Adam)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


학습 중간중간 검증 성능을 확인할 mAP 계산 함수. iou_thresholds를 0.75~0.95로 좁게 잡아서 좀 빡빡하게 채점함(캐글 채점 기준이랑 맞추려는 목적). torchmetrics의 MeanAveragePrecision을 사용.

In [ ]:
#@title mAP 계산 함수 (IoU threshold 0.75~0.95)
iou_thresholds = [0.75, 0.80, 0.85, 0.90, 0.95]

def evaluate_map(model, data_loader, device):
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", iou_thresholds=iou_thresholds)

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            preds = model(images)

            preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
            targets_cpu = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]

            metric.update(preds_cpu, targets_cpu)

    result = metric.compute()
    model.train()
    return result

실제로 30 epoch 학습을 돌리는 셀. epoch마다 train loss 계산하고, evaluate_map으로 val mAP까지 같이 찍어서 학습이 잘 되고 있는지 바로바로 확인 가능. 30 epoch 다 도는 데 코랩 T4 기준으로 대략 25분 정도 걸림.

In [ ]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_train_loss += losses.item()

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    lr_scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)

    val_map = evaluate_map(model, val_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} "
          f"| mAP[.75:.95]: {val_map['map'].item():.4f} "
          f"| mAP@75: {val_map['map_75'].item():.4f}")

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
Evaluating: 100%|██████████| 12/12 [00:06<00:00,  1.73it/s]


Epoch 1/30 | Train Loss: 1.3169 | mAP[.75:.95]: 0.0188 | mAP@75: 0.0581


Evaluating: 100%|██████████| 12/12 [00:07<00:00,  1.66it/s]


Epoch 2/30 | Train Loss: 0.6875 | mAP[.75:.95]: 0.1503 | mAP@75: 0.3293


Evaluating: 100%|██████████| 12/12 [00:07<00:00,  1.64it/s]


...

Epoch 28/30 | Train Loss: 0.2518 | mAP[.75:.95]: 0.5787 | mAP@75: 0.8328


Evaluating: 100%|██████████| 12/12 [00:07<00:00,  1.67it/s]


Epoch 29/30 | Train Loss: 0.2475 | mAP[.75:.95]: 0.5925 | mAP@75: 0.8099


Evaluating: 100%|██████████| 12/12 [00:08<00:00,  1.47it/s]


Epoch 30/30 | Train Loss: 0.2462 | mAP[.75:.95]: 0.5669 | mAP@75: 0.7813


개선할 수 있는 점
1. 박스의 개수에 제한을 하나도 걸지 않았음
- 모델을 개선하기 시작하는 경우, train / test 데이터에 알약이 3개 아니면 4개만 있다는 점을 확인하고, BoundingBox의 개수를 제한할 수 있음
2. 그냥 미션 7에서 활용한 VGGNet이랑 SSD 그대로 활용했음
- 사실 VGGNet에 SSD는 너무 오래된 전통적인 모델이라 다른 조합을 생각해볼 수 있음
- 다음의 조합도 다 실행해 봐야겠음

![image.png](attachment:image.png)

| Method | Backbone | AP | AP₅₀ | AP₇₅ | APₛ | APₘ | APₗ |
|---|---|---:|---:|---:|---:|---:|---:|
| **Two-stage methods** |  |  |  |  |  |  |  |
| Faster R-CNN+++ [16] | ResNet-101-C4 | 34.9 | 55.7 | 37.4 | 15.6 | 38.7 | 50.9 |
| Faster R-CNN w FPN [20] | ResNet-101-FPN | 36.2 | 59.1 | 39.0 | 18.2 | 39.0 | 48.2 |
| Faster R-CNN by G-RMI [17] | Inception-ResNet-v2 [34] | 34.7 | 55.5 | 36.7 | 13.5 | 38.1 | 52.0 |
| Faster R-CNN w TDM [32] | Inception-ResNet-v2-TDM | 36.8 | 57.7 | 39.2 | 16.2 | 39.8 | 52.1 |
| **One-stage methods** |  |  |  |  |  |  |  |
| YOLOv2 [27] | DarkNet-19 [27] | 21.6 | 44.0 | 19.2 | 5.0 | 22.4 | 35.5 |
| SSD513 [22, 9] | ResNet-101-SSD | 31.2 | 50.4 | 33.3 | 10.2 | 34.5 | 49.8 |
| DSSD513 [9] | ResNet-101-DSSD | 33.2 | 53.3 | 35.2 | 13.0 | 35.4 | 51.1 |
| RetinaNet (ours) | ResNet-101-FPN | 39.1 | 59.1 | 42.3 | 21.8 | 42.7 | 50.2 |
| RetinaNet (ours) | ResNeXt-101-FPN | **40.8** | **61.1** | **44.1** | **24.1** | **44.2** | 51.2 |

제출용 전체 추론 전에, test 이미지 중 20장만 미리 뽑아서 눈으로 확인해보는 셀. 실제 제출 파일은 이 20장이 아니라 test_images 폴더 전체로 따로 만듦(맨 아래 Submission CSV 생성 셀에서).

In [ ]:
#@title Test 이미지 경로 설정 및 랜덤 샘플링
import random

test_image_dir = glob.glob(os.path.join(path, "test_images", "*.png"))
print(f"Test 이미지 수: {len(test_image_dir)}")

random.seed(42)
test_sample_paths = random.sample(test_image_dir, min(20, len(test_image_dir)))
test_sample_stems = [os.path.splitext(os.path.basename(p))[0] for p in test_sample_paths]

Test 이미지 수: 842


test 이미지는 정답 라벨이 없어서 Dataset도 따로 만들어야 함. image랑 stem(파일명)만 리턴하는 간단한 버전.

In [ ]:
#@title TestDataset 정의
class TestDataset(Dataset):
    def __init__(self, image_dir, stems, transforms):
        self.image_dir = image_dir
        self.stems = stems
        self.transforms = transforms

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        image = Image.open(os.path.join(self.image_dir, f"{stem}.png")).convert("RGB")
        image = tv_tensors.Image(image)

        if self.transforms:
            image = self.transforms(image)

        return image, stem

test_dataset = TestDataset(
    image_dir=os.path.join(path, "test_images"),
    stems=test_sample_stems,
    transforms=val_test_transformer
)

test_loader = DataLoader(
    test_dataset, batch_size=4, shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

모델이 예측한 박스를 이미지 위에 그려서 보여주는 함수. score_thresh보다 점수 낮은 박스는 안 그리게 걸러냄 — 여기 기본값 0.5는 눈으로 확인할 때 쓰는 값이고, 실제 제출 파일 만들 때 쓰는 threshold(0.3)랑은 다른 값임.

In [ ]:
#@title Define method : visualize_prediction
def visualize_prediction(image, prediction, classes, score_thresh=0.5):
    """
    image (torch.Tensor): 추론에 사용된 이미지 (C, H, W).
    prediction (dict): boxes, labels, scores 포함.
    classes (list): index -> 약 이름 매핑 리스트 (0번 = background).
    """
    image = image.permute(1, 2, 0).cpu().numpy()

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(image)

    for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
        if score > score_thresh:
            x_min, y_min, x_max, y_max = box.tolist()
            width, height = x_max - x_min, y_max - y_min

            rect = patches.Rectangle(
                (x_min, y_min), width, height,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x_min, max(y_min - 5, 0),
                f"{classes[label]} ({score:.2f})",
                color="white", fontsize=9,
                bbox=dict(facecolor="red", alpha=0.6, pad=1)
            )

    ax.axis("off")
    plt.show()

위에서 뽑은 20장으로 실제 모델 예측 결과를 눈으로 확인하는 셀. 박스가 알약 위치에 잘 맞는지, 라벨이 말이 되는지 여기서 먼저 감으로 체크하고 넘어감.

In [ ]:
#@title Test 이미지 20장 추론 및 시각화
model.eval()

with torch.no_grad():
    for images, stems in tqdm(test_loader, desc="Test Inference"):
        images = [img.to(device) for img in images]
        predictions = model(images)

        for img, pred, stem in zip(images, predictions, stems):
            print(f"파일명: {stem}")
            visualize_prediction(img.cpu(), pred, classes)

Test Inference:   0%|          | 0/5 [00:00<?, ?it/s]WARNING:matplotlib.image:Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.16267046..1.0000001].


파일명: 1190


파일명: 948


/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 45349 (\N{HANGUL SYLLABLE NEG}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 49888 (\N{HANGUL SYLLABLE SIN}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 50640 (\N{HANGUL SYLLABLE E}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 54532 (\N{HANGUL SYLLABLE PEU}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/loc

파일명: 210


...

파일명: 1390


파일명: 941


파일명: 957


/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 44032 (\N{HANGUL SYLLABLE GA}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 47161 (\N{HANGUL SYLLABLE RYEONG}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.13/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 54028 (\N{HANGUL SYLLABLE PA}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)


Test Inference: 100%|██████████| 5/5 [00:12<00:00,  2.42s/it]


기존 다른 브랜치(feature/initial-modeling, 커밋 2470646)의 제출 코드는 score_thresh를 0.05로 거의 걸러지지 않게 두고 scores.argsort()[:4]로 점수 순위와 상관없이 무조건 상위 4개 박스만 제출했습니다.
그런데 실제 test 이미지는 알약이 3개인 것도 있고 5개인 것도 있어서, 개수를 4개로 강제하면 3개짜리 이미지에는 없는 박스가 하나 끼어들고(오탐) 5개짜리 이미지에는 있는 박스가 하나 빠지는 문제가 있었습니다.
그래서 이 셀은 개수를 강제하지 않고 score_thresh = 0.3 기준으로만 거르도록 수정했습니다. (filter_submission.py로 사후 필터링했을 때 threshold 0.3이 0.5보다 결과가 좋았던 값을 그대로 가져왔습니다.)
단, threshold를 넘는 박스가 하나도 없는 이미지는 제출 파일에서 통째로 빠질 수 있어서, 그런 경우에만 예외적으로 점수가 가장 높은 박스 1개를 남기도록 안전장치를 뒀습니다.

In [ ]:
#@title Submission CSV 생성 (score_thresh 필터링, top-4 강제 X)
# 왜 이렇게 짰는지는 위 텍스트 셀 참고
model.eval()

# test_images 폴더 전체 사용 (위 20장 샘플용 test_loader랑 다름)
all_test_paths = glob.glob(os.path.join(path, "test_images", "*.png"))
all_test_stems = [os.path.splitext(os.path.basename(p))[0] for p in all_test_paths]
print(f"제출용 test 이미지 수: {len(all_test_stems)}")

submission_test_dataset = TestDataset(
    image_dir=os.path.join(path, "test_images"),
    stems=all_test_stems,
    transforms=val_test_transformer
)
submission_test_loader = DataLoader(
    submission_test_dataset, batch_size=8, shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

def extract_image_id(stem):
    return int(stem)

SCORE_THRESH = 0.3  # filter_submission.py 실험값 (0.5보다 0.3이 나았음)

rows = []
by_image_all = {}  # 안전장치용, threshold 걸기 전 예측 전부 보관

with torch.no_grad():
    for images, stems in tqdm(submission_test_loader, desc="Submission Inference"):
        images = [img.to(device) for img in images]
        predictions = model(images)

        for pred, stem in zip(predictions, stems):
            image_id = extract_image_id(stem)
            boxes = pred["boxes"].cpu()
            labels = pred["labels"].cpu()
            scores = pred["scores"].cpu()

            image_rows = []
            for box, label, score in zip(boxes, labels, scores):
                x_min, y_min, x_max, y_max = box.tolist()
                bbox_w = x_max - x_min
                bbox_h = y_max - y_min
                category_id = label_to_category[label.item()]  # 내부 라벨 -> 원본 category_id

                row = [image_id, category_id,
                       round(x_min, 2), round(y_min, 2),
                       round(bbox_w, 2), round(bbox_h, 2),
                       round(score.item(), 4)]
                image_rows.append(row)

            by_image_all[image_id] = image_rows
            rows.extend(r for r in image_rows if r[-1] >= SCORE_THRESH)

# threshold 넘는 박스가 하나도 없는 이미지는 최고 score 박스 1개만 살려둠
kept_images = {r[0] for r in rows}
missing_images = set(by_image_all.keys()) - kept_images
for image_id in missing_images:
    if by_image_all[image_id]:
        best = max(by_image_all[image_id], key=lambda r: r[-1])
        rows.append(best)
        print(f"  안전장치 적용: image_id={image_id}는 threshold({SCORE_THRESH}) 이상 박스가 없어 "
              f"최고 score({best[-1]:.4f}) 박스 1개만 남김")

# annotation_id 재부여 (1부터 순차)
final_rows = [[i, *row] for i, row in enumerate(rows, start=1)]

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"])
    writer.writerows(final_rows)

print(f"총 {len(final_rows)}개 row 저장 완료 -> submission.csv (score_thresh={SCORE_THRESH}, 개수 강제 없음)")


제출용 test 이미지 수: 842


Submission Inference: 100%|██████████| 106/106 [02:08<00:00,  1.21s/it]

총 8448개 row 저장 완료 -> submission.csv (score_thresh=0.3, 이미지당 개수 강제 없음)
